# 6.4 预训练 Transformer 迁移：AST 三模式

基于AudioSet预训练的AST（Audio Spectrogram Transformer；本章下游模型含86,191,878个参数）
在 CTMP 6 类数据集上做迁移学习，对比三种迁移模式：
Feature Extraction / LoRA / Full Fine-tuning。

评估协议（与 6.1b/6.2/6.3 一致）：
- **选择**：train split 训练 → val split 选择 best epoch / 最佳超参
- **内部**：锁定 best epoch 后，仅在 test split 做最终评估
- **外部**：同一模型在 external_test（ChMusic）上推理
- **划分敏感性**：在 3 份冻结的 train/val/test 划分上评估，取均值和总体标准差；external_test固定不变


## 1. 环境准备

约定环境变量：
- `HF_ENDPOINT=https://hf-mirror.com`：HuggingFace 国内镜像
- `HF_HUB_OFFLINE=1` / `TRANSFORMERS_OFFLINE=1`：禁止 Hub 网络请求，所需文件必须已在本地

AST 权重已经预先 curl 到 `outputs/checkpoints/ast/` 本地目录，开启 offline 模式后即使没有网络也能完整跑通。


In [ ]:
import os
os.environ.setdefault('HF_ENDPOINT', 'https://hf-mirror.com')
os.environ.setdefault('HF_HUB_OFFLINE', '1')
os.environ.setdefault('TRANSFORMERS_OFFLINE', '1')

import sys
from pathlib import Path

# 路径推断：从 cwd 向上找含 CODE/datasets 的目录；PROJECT_ROOT 指向 CODE/
_p = Path.cwd()
while not (_p / "CODE" / "datasets").exists():
    _parent = _p.parent
    if _parent == _p:
        raise FileNotFoundError("未找到项目根目录（包含 CODE/datasets 的目录），请在项目内运行本 Notebook")
    _p = _parent
PROJECT_ROOT = _p / "CODE"  # CODE/
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('PROJECT_ROOT:', PROJECT_ROOT)

import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['font.sans-serif'] = [
    'Hiragino Sans GB', 'PingFang SC', 'Arial Unicode MS', 'STHeiti', 'Heiti TC',
    'Microsoft YaHei', 'SimHei', 'Noto Sans CJK SC', 'DejaVu Sans',
]
matplotlib.rcParams['axes.unicode_minus'] = False
print('matplotlib:', matplotlib.__version__)


### 1.1 依赖与数据齐备性检查

`transformers`是本节额外使用的依赖。AST权重（346,404,948字节）需预先下载到
`outputs/checkpoints/ast/`，offline 模式开启后才能离线跑通。


In [ ]:
from chapter06._common import check_environment

check_environment(notebook='06_4_ast', require_ctmp=True)


In [ ]:
import time

import numpy as np
import pandas as pd
import torch

from chapter06._common import (
    add_recall_colorbar,
    get_device,
    plot_confusion_matrix,
    setup_chinese_font,
)
from chapter06._common.ctmp_loader import (
    CTMP_CLASSES,
    build_label_map,
    get_ctmp_output_dir,
)
from chapter06._common.device_utils import auto_batch_size
from chapter06.pretrained_transformer import (
    TransferAST,
    ASTTrainConfig,
    run_ast_experiment,
    TRANSFER_MODES,
    apply_lora,
    freeze_backbone,
    fine_tune_all,
    count_trainable,
    compact_sweep_configs,
    run_ast_config_grid,
    selected_config_table,
    select_best_specs_by_val,
    summarize_config_results,
    summarize_selected_methods,
)

setup_chinese_font()

OUT_DIR = PROJECT_ROOT / 'chapter06' / 'pretrained_transformer' / 'outputs'
FIG_DIR = OUT_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

SEEDS = [0, 1, 2]
# 变量名沿用代码接口；0/1/2 实际表示三份冻结 manifest 的编号。
LABEL_MAP = build_label_map()
CLASS_NAMES = list(CTMP_CLASSES)
N_CLASSES = len(CLASS_NAMES)

print(f'CTMP 类别: {CLASS_NAMES} ({N_CLASSES} 类)')
print(f'CTMP 输出目录: {get_ctmp_output_dir()}')
print(f'device: {get_device()} | batch_size: {auto_batch_size()}')
print(f'torch: {torch.__version__}')


## 2. AST 架构介绍

### 跨模态预训练链条

AST先用ImageNet-2012上训练的DeiT参数初始化图像块嵌入、特殊标记和
Transformer编码器，再调整输入层与位置编码以接收log-mel特征，随后在AudioSet
上训练。这里的“跨模态迁移”描述参数初始化路径，不等于已经识别出具体迁移了
哪些视觉或声学因素。

```
ImageNet-2012（DeiT视觉预训练）
    ↓ 迁移 Transformer blocks 与特殊标记
    ↓ 调整 patch embedding / positional embedding 以适配 mel 时频形状
AudioSet（约208万条官方清单、527类；片段目标长度通常为10秒）
    ↓
本节下游任务：CTMP 6 类乐器分类
```

### 与 CNN14 的对比

| 维度 | CNN14 (6.3) | AST (6.4) |
|:---|:---|:---|
| 参数量 | 约8100万（统计口径见6.3） | 86,191,878（本章下游模型） |
| 架构 | 6 个卷积块 + GAP + FC | DeiT-Base：12 层 Transformer + 双特殊标记平均池化 |
| 感受野 | 局部 → 全局（堆叠卷积） | 全局（patch 间 self-attention） |
| 输入 | 32 kHz波形 → 内置log-mel | 16 kHz波形 → log-mel patch 16×16 |
| 预训练 | AudioSet 直接训练 | ImageNet → AudioSet 跨模态 |


In [ ]:
# 实例化 AST 模型，确认参数量与三种模式可训练参数
model_fe = TransferAST(n_classes=N_CLASSES); freeze_backbone(model_fe)
print(f'Feature Extraction 可训练参数: {count_trainable(model_fe):,}')

model_lora = TransferAST(n_classes=N_CLASSES); apply_lora(model_lora)
print(f'LoRA 可训练参数: {count_trainable(model_lora):,}')

model_full = TransferAST(n_classes=N_CLASSES); fine_tune_all(model_full)
print(f'Full Fine-tune 可训练参数: {count_trainable(model_full):,}')

del model_fe, model_lora, model_full


## 3. 三种迁移模式

| 模式 | 策略 | 可训练参数 | 与 6.3 CNN14 对应 |
|:---|:---|:---:|:---|
| Feature Extraction | 冻结 ASTModel，只训练 `Linear(768, 6)` | 4,614 | 更新范围相近 |
| LoRA | attention `q_proj/v_proj` 注入低秩adapter；本节比较r=4/8/16 | 约15万至59万 | 无直接对应策略 |
| Full Fine-tuning | 解冻所有层；本节 sweep lr=5e-6/1e-5/2e-5 | 86,191,878 | 更新范围相近 |

实现上，Feature Extraction 先在 `eval + no_grad` 下抽取 frozen AST 的双特殊标记平均embedding，再用训练集统计量标准化，并在固定 embedding 上训练线头。
该路径对应“冻结且置于评估态的骨干 + 可训练分类头”，同时减少重复的骨干前向计算。

### LoRA分解与参数量

LoRA 把权重更新 ΔW 分解为两个低秩矩阵的乘积 BA（A: r×d，B: d×r，r≪d），
训练矩阵 A、B 而冻结原始 W。当秩 r 远小于原矩阵维度时，可训练参数由完整矩阵的 $dk$ 降为 $r(d+k)$；
更新后的有效权重仍为 $W + BA$，实际结果取决于秩、缩放系数、学习率、数据和训练协议。

三种模式改变可训练参数范围、优化器参数组和计算成本。LoRA在原权重冻结时增加低秩适配器，因而可训练参数少于全量微调；
三种策略在CTMP上的排序由后续验证集选择和测试结果给出，不能仅凭参数量预先确定。


## 4. 训练与评估（验证集选择与有限网格）

### 本节训练配置

| 组件 | 值 | 动机 |
|:---|:---|:---|
| optimizer | Adam | 本节固定采用 Adam；`weight_decay` 按当前 PyTorch Adam 定义作为与梯度耦合的 L2 项，不等同于 AdamW |
| schedule | linear warmup + linear decay | 前 10% steps 把 lr 从 0 线性升到目标，随后线性降到 0 |
| grad clip | 1.0 | 在优化器更新前把可训练参数的梯度总范数截断到 1.0 |
| label smoothing | 0.1 | 把 one-hot 目标替换为带 0.1 平滑系数的目标分布 |
| Feature Extraction | standardized frozen pooled embedding + CPU linear head | 固定 AST 表征后只优化 4,614 个 head 参数；按模型定义平均 CLS 与 distillation token，标准化所得 embedding，并在 CPU 上训练线性头 |
| LoRA/FullFT batch | micro-batch 8 + grad accumulation 4 | 每次前向/反向处理 8 个样本，每 4 个微批次更新一次；减少单次激活张量规模 |
| best epoch | LoRA/FullFT: val loss；Feature Extraction: val macro-F1 | 不使用 test loss 选择模型；LoRA和FullFT候选配置按三份冻结划分的平均val macro-F1选择 |
| LoRA sweep | r=4/8/16，alpha=2r | 选择 val macro-F1 最好的 LoRA 配置 |
| FullFT sweep | lr=5e-6/1e-5/2e-5 | 选择 val macro-F1 最好的 FullFT 配置 |
| epochs | 最多 20，patience=5 | LoRA/FullFT 的 val loss 或 Feature Extraction 的 val macro-F1 连续 5 轮未改善时提前停止；缓存独立放在 `outputs/run_cache/ast_val/` |

LoRA和FullFT候选配置先按三份划分的平均验证macro-F1排序，再以平均验证Acc处理并列；
若两项仍完全相同，则按预先声明的候选顺序选择。当前候选顺序对LoRA按较低秩在前，
对FullFT按较低学习率在前。该规则在查看test或external_test结果之前确定。

**结果级缓存**：每个 `(config_key, device, seed)` 跑完后落盘到
`outputs/run_cache/ast_val/{config_key}_{device}_seed{seed}.pkl`。cell 重跑或 kernel 重启时，
如果 cache 的训练协议、超参、设备、PyTorch与Transformers版本、checkpoint大小、选择 split 与 cache policy 全部匹配，就直接跳过训练，
只重新汇总表格和图片；否则会把该文件判定为 stale 并重跑对应 run。
这是完成 run 之后的结果缓存，不是 epoch 级 checkpoint，因此训练中途被打断时，
未完成的 run 会重跑。Feature Extraction 会额外检查 embedding/head 策略；
LoRA/FullFT 会检查具体 rank、alpha、dropout、lr、batch 与 grad accumulation。
旧目录 `outputs/run_cache/ast/` 是 v2/test-loss 选择策略的废弃缓存，新版 notebook 和 `run_all_ast.py` 不读取。确认 `ast_val/` 完整后可以删除。


In [ ]:
RUN_CACHE = OUT_DIR / 'run_cache' / 'ast_val'
SWEEP_SPECS = compact_sweep_configs(epochs=20, batch_size=32, tune_batch_size=8)

all_results = run_ast_config_grid(SWEEP_SPECS, SEEDS, RUN_CACHE)
df_sweep = summarize_config_results(all_results)
df_sweep.to_csv(OUT_DIR / 'ast_hparam_sweep.csv', index=False)
print('written:', (OUT_DIR / 'ast_hparam_sweep.csv').resolve())

selected_specs = select_best_specs_by_val(SWEEP_SPECS, df_sweep)
df_selected_configs = selected_config_table(selected_specs, df_sweep)
df_selected_configs.to_csv(OUT_DIR / 'ast_selected_configs.csv', index=False)
print('written:', (OUT_DIR / 'ast_selected_configs.csv').resolve())

df_sweep[['family', 'display_name', 'val_f1', 'val_acc', 'ext_acc', 'Gap (Acc)']]


## 5. 主结果表：3 份冻结划分的均值 ± 总体标准差

内部（test）与外部（external_test）的 Acc 与 macro-F1 并列，
Gap = 内部 − 外部。LoRA / FullFT 先按 val macro-F1 选择最佳配置，再进入主表。
CSV schema 与 6.3 对齐（中文列名 + mean±std 字符串）。


In [ ]:
df_summary = summarize_selected_methods(all_results, selected_specs)
df_summary.to_csv(OUT_DIR / 'ast_method_comparison.csv', index=False)
print('written:', (OUT_DIR / 'ast_method_comparison.csv').resolve())
df_summary


## 6. 候选配置汇总与训练曲线（冻结划分0，入选配置）

先看全部候选配置的外部 Acc，再画入选配置的训练曲线。
训练曲线只展示 train / val：验证集用于选择 best epoch，test/external_test 不参与训练过程。

Feature Extraction 的曲线来自标准化固定 pooled embedding 上的 CPU 线性头训练；
`train_loss/train_acc` 是每轮结束后在train split上的评估结果，与LoRA和FullFT的
端到端训练路径不同。曲线可用于检查优化过程，但不能单独解释测试集差异。


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
plot_df = df_sweep.sort_values(['mode', 'ext_acc_mean'])
colors = plot_df['mode'].map({
    'feature_extraction': '0.35',
    'lora': '0.55',
    'fine_tune': '0.75',
})
ax.barh(plot_df['display_name'], plot_df['ext_acc_mean'], color=colors)
ax.set_xlabel('external Acc')
ax.set_xlim(0, 1.0)
ax.set_title('AST 有限候选网格 · 外部准确率')
fig.tight_layout()
fig.savefig(FIG_DIR / 'ast_sweep_external_acc.png', dpi=600, bbox_inches='tight')
plt.show()

def plot_history(history, title, ax_loss, ax_metric):
    epochs = range(1, len(history['train_loss']) + 1)
    ax_loss.plot(epochs, history['train_loss'], label='train', color='0.3')
    ax_loss.plot(epochs, history['val_loss'], label='val', color='0.6', linestyle='--')
    ax_loss.set_xlabel('epoch'); ax_loss.set_ylabel('loss')
    ax_loss.set_title(f'{title} · loss'); ax_loss.legend()
    ax_metric.plot(epochs, history['val_acc'], label='accuracy', color='0.3')
    ax_metric.plot(epochs, history['val_f1'], label='macro-F1', color='0.6', linestyle='--')
    ax_metric.set_xlabel('epoch'); ax_metric.set_ylabel('指标')
    ax_metric.set_title(f'{title} · val 指标'); ax_metric.legend()

seed0_results = [r for r in all_results if r['seed'] == 0]
seed0_by_key = {r['config_key']: r for r in seed0_results}

fig, axes = plt.subplots(len(selected_specs), 2, figsize=(12, 4 * len(selected_specs)))
for row, spec in enumerate(selected_specs):
    plot_history(seed0_by_key[spec['key']]['history'], spec['display_name'],
                 axes[row, 0], axes[row, 1])
fig.tight_layout()
fig.savefig(FIG_DIR / 'training_curves.png', dpi=600, bbox_inches='tight')
plt.show()


## 7. 混淆矩阵：内部与外部（划分0，按验证集选择模式）

LoRA 与 Full Fine-tune 的配置已按验证集选定；三种入选模式之间再按 3 份划分的
平均 val macro-F1 选择一种，并绘制其划分 0 的内部 test 与外部 external_test
混淆矩阵。平均 val macro-F1 与平均 val Acc 均完全相同时，按预先声明的
Feature Extraction、LoRA、Full Fine-tune顺序确定展示策略。最终测试集不参与选择。


In [ ]:
selected_keys = [spec['key'] for spec in selected_specs]
selected_summary = df_sweep[df_sweep['config_key'].isin(selected_keys)].copy()
selected_order = {spec['key']: idx for idx, spec in enumerate(selected_specs)}
selected_summary['_selected_order'] = selected_summary['config_key'].map(selected_order)
best_key = selected_summary.sort_values(
    ['val_f1_mean', 'val_acc_mean', '_selected_order'],
    ascending=[False, False, True], kind='mergesort',
).iloc[0]['config_key']
best = seed0_by_key[best_key]
best_title = best['config_display']
best_val_f1 = selected_summary.loc[
    selected_summary['config_key'] == best_key, 'val_f1_mean'
].iloc[0]
print(f'按平均 val macro-F1 选择: {best_title} | val_f1={best_val_f1:.3f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
plot_confusion_matrix(
    axes[0], best['y_test'], best['pred_test'],
    class_names=CLASS_NAMES, title=f'{best_title} — 内部 test',
    labels=list(range(N_CLASSES)),
)
plot_confusion_matrix(
    axes[1], best['y_ext'], best['pred_ext'],
    class_names=CLASS_NAMES, title=f'{best_title} — 外部 external_test',
    labels=list(range(N_CLASSES)),
)
fig.subplots_adjust(wspace=0.05)
add_recall_colorbar(fig, [axes[0], axes[1]])
fig.savefig(FIG_DIR / 'confusion_internal_vs_external.png', dpi=600, bbox_inches='tight')
plt.show()


## 8. 与 6.1/6.2/6.3 全量对比

读 6.3 pretrained_cnn 的 `full_comparison.csv`（已含 6.1b + 6.2 + 6.3），
追加本节 AST 三模式后保存到 `pretrained_transformer/outputs/full_comparison.csv`。


In [ ]:
cnn_csv = PROJECT_ROOT / 'chapter06' / 'pretrained_cnn' / 'outputs' / 'full_comparison.csv'
if not cnn_csv.exists():
    raise FileNotFoundError(
        '缺少6.3结果：请先执行06_3_pretrained_cnn.ipynb，再生成全章比较表'
    )
df_prev = pd.read_csv(cnn_csv, dtype=str)
df_all = pd.concat([df_prev, df_summary], ignore_index=True)

df_all.to_csv(OUT_DIR / 'full_comparison.csv', index=False)
print('written:', (OUT_DIR / 'full_comparison.csv').resolve())
df_all


## 9. 结论

训练过程用val split选择最佳轮次，并分别对LoRA秩和Full Fine-tuning学习率做有限网格
搜索。当前网格共7个配置×三份冻结划分：Feature Extraction、LoRA r=4/8/16、
FullFT lr=5e-6/1e-5/2e-5。`df_sweep`、`df_selected_configs`和`df_summary`分别记录
全部候选配置、按验证集入选的配置和最终内部/外部测试汇总。

解读时注意两条边界：

- **调参只看 val**：test 和 external_test 不参与 best epoch 或最佳超参选择，只做最终评估。
- **最优只限当前网格**：当前实验只比较列出的有限配置，不表示全局最优。若要扩大结论范围，
  可使用 `extended_sweep_configs()` 增加候选配置，并增加独立冻结划分或外部数据来源。

内部 test 接近满分并不等于外部数据上的表现相同。LoRA 与 FullFT 的 sweep 用于比较
当前网格内的验证集结果。由于只有 3 份冻结划分，这些结果仍不足以区分超参数效应与划分波动。

**下一步（6.4 第二半）**：CLAP zero-shot——不使用CTMP训练集更新参数，以文本提示指定候选类别。
